# SkillForge — Document Roundtrip Tutorial

**Convert between PDF, HTML, Markdown, and AIF using real Wikipedia articles.**

This notebook walks through three document-roundtrip pipelines:

1. **HTML → AIF → HTML / LML / Markdown** (via Python API)
2. **Markdown → AIF → Markdown** (via Python API)
3. **PDF → AIF → PDF / Markdown** (via the `aif` CLI, since PDF support lives in a heavy Rust crate)

All examples use real pages fetched live from Wikipedia. Run this top-to-bottom.

## Prerequisites

```bash
pip install aif-skillforge   # Python bindings
```

For the PDF section you also need the Rust CLI:

```bash
cargo install --path crates/aif-cli    # or `cargo build --release -p aif-cli`
```

In [1]:
import urllib.request, json, os, subprocess
from pathlib import Path
import skillforge

WORK = Path('/tmp/aif_roundtrip_demo'); WORK.mkdir(exist_ok=True)
UA = {'User-Agent': 'AIF-Tutorial/1.0'}

def fetch(url: str) -> bytes:
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read()

def fmt_bytes(n: int) -> str:
    return f'{n/1024:.1f} KB' if n < 1_000_000 else f'{n/1024/1024:.2f} MB'

print(f'Working directory: {WORK}')
print(f'skillforge functions: {len([f for f in dir(skillforge) if not f.startswith("_")])}')

Working directory: /tmp/aif_roundtrip_demo
skillforge functions: 17


## 1 · HTML → AIF → HTML / Markdown / LML

We fetch the **Photosynthesis** Wikipedia article as raw HTML, strip chrome (nav/headers/footers), and import it into AIF. Then we render it back to HTML, Markdown, and LML Aggressive to see the size savings.

In [2]:
# Fetch Photosynthesis as raw HTML from the Wikipedia REST API
html = fetch('https://en.wikipedia.org/api/rest_v1/page/html/Photosynthesis').decode('utf-8')
(WORK / 'photosynthesis.html').write_text(html)
print(f'Raw HTML: {fmt_bytes(len(html))}')
print(f'First 200 chars: {html[:200]}...')

Raw HTML: 763.3 KB
First 200 chars: <!DOCTYPE html>
<html prefix="dc: http://purl.org/dc/terms/ mw: http://mediawiki.org/rdf/" about="https://en.wikipedia.org/wiki/Special:Redirect/revision/1346031235"><head prefix="mwr: https://en.wiki...


In [3]:
# HTML -> AIF IR (strip chrome via readability)
ir_json = skillforge.clean_html(html)
ir = json.loads(ir_json)
(WORK / 'photosynthesis.ir.json').write_text(ir_json)

print(f'AIF IR: {len(ir["blocks"])} blocks')
print(f'Metadata: {ir["metadata"]}')
print()
print('Block-type distribution:')
from collections import Counter
counts = Counter(b['kind']['type'] for b in ir['blocks'])
for t, n in counts.most_common():
    print(f'  {t:20s} {n}')

AIF IR: 14 blocks
Metadata: {'_aif_import_mode': 'readability', '_aif_source_format': 'html', 'title': 'Photosynthesis'}

Block-type distribution:
  Section              14


In [4]:
# AIF IR -> HTML / Markdown / LML Aggressive
html_rt = skillforge.render(ir_json, 'html')
md_rt   = skillforge.render(ir_json, 'markdown')
lml_rt  = skillforge.render(ir_json, 'lml-aggressive')

(WORK / 'photosynthesis.roundtrip.html').write_text(html_rt)
(WORK / 'photosynthesis.roundtrip.md').write_text(md_rt)
(WORK / 'photosynthesis.roundtrip.lml').write_text(lml_rt)

print(f'{"Format":25s} {"Size":>12s} {"vs raw HTML":>14s}')
print('-' * 55)
print(f'{"Raw HTML (baseline)":25s} {fmt_bytes(len(html)):>12s} {"—":>14s}')
print(f'{"AIF HTML (roundtrip)":25s} {fmt_bytes(len(html_rt)):>12s} {(1-len(html_rt)/len(html))*100:>13.0f}%')
print(f'{"AIF Markdown":25s} {fmt_bytes(len(md_rt)):>12s} {(1-len(md_rt)/len(html))*100:>13.0f}%')
print(f'{"AIF LML Aggressive":25s} {fmt_bytes(len(lml_rt)):>12s} {(1-len(lml_rt)/len(html))*100:>13.0f}%')
print()
print('--- First 400 chars of LML Aggressive ---')
print(lml_rt[:400])

Format                            Size    vs raw HTML
-------------------------------------------------------
Raw HTML (baseline)           763.3 KB              —
AIF HTML (roundtrip)          237.5 KB            69%
AIF Markdown                  195.2 KB            74%
AIF LML Aggressive            191.3 KB            75%

--- First 400 chars of LML Aggressive ---
#_aif_import_mode: readability
#_aif_source_format: html
#title: Photosynthesis
# 
@fig(src=//upload.wikimedia.org/wikipedia/commons/thumb/5/55/Photosynthesis_en.svg/500px-Photosynthesis_en.svg.png, w=380, h=380(id=mwCg)): Schematic of photosynthesis in plants. The carbohydrates (./Carbohydrate) produced are stored in or used by the plant.

@fig(src=//upload.wikimedia.org/wikipedia/commons/thumb/4


## 2 · Markdown → AIF → Markdown

Direct markdown roundtrip. We write a small structured document, import it to AIF IR, and render back. Block structure is preserved; inline formatting is preserved.

In [5]:
md_source = '''# Photosynthesis

Photosynthesis is a **biological process** used by *plants* to convert light energy into chemical energy.

## Overview

It takes place in the chloroplasts of plant cells.

1. Light-dependent reactions capture sunlight
2. Light-independent reactions (Calvin cycle) fix carbon
3. Products: glucose and oxygen

The chemical equation is `6CO2 + 6H2O + light -> C6H12O6 + 6O2`.

## History

Photosynthesis was first understood by [Jan Ingenhousz](https://en.wikipedia.org/wiki/Jan_Ingenhousz).

> "All plants exhale a fluid that disappears on heating."

| Organism | Type | Habitat |
| --- | --- | --- |
| Spinach | C3 plant | Terrestrial |
| Corn | C4 plant | Terrestrial |
| Cyanobacteria | Prokaryote | Aquatic |
'''
print(f'Source Markdown: {len(md_source)} bytes, {len(md_source.splitlines())} lines')

Source Markdown: 729 bytes, 25 lines


In [6]:
# MD -> AIF IR
ir_json = skillforge.import_markdown(md_source)
ir = json.loads(ir_json)
print(f'Imported: {len(ir["blocks"])} blocks')
for b in ir['blocks']:
    print(f'  - {b["kind"]["type"]}')

# AIF IR -> Markdown
md_rt = skillforge.render(ir_json, 'markdown')
print()
print(f'Roundtrip: {len(md_rt)} bytes, {len(md_rt.splitlines())} lines')
print()
print('--- Roundtripped markdown ---')
print(md_rt)

Imported: 10 blocks
  - Section
  - Paragraph
  - Section
  - Paragraph
  - List
  - Paragraph
  - Section
  - Paragraph
  - BlockQuote
  - Table

Roundtrip: 748 bytes, 27 lines

--- Roundtripped markdown ---
# Photosynthesis

## Photosynthesis

Photosynthesis is a **biological process** used by *plants* to convert light energy into chemical energy.

## Overview

It takes place in the chloroplasts of plant cells.

1. Light-dependent reactions capture sunlight
2. Light-independent reactions (Calvin cycle) fix carbon
3. Products: glucose and oxygen

The chemical equation is `6CO2 + 6H2O + light -> C6H12O6 + 6O2`.

## History

Photosynthesis was first understood by [Jan Ingenhousz](https://en.wikipedia.org/wiki/Jan_Ingenhousz).

> "All plants exhale a fluid that disappears on heating."

| Organism | Type | Habitat |
| --- | --- | --- |
| Spinach | C3 plant | Terrestrial |
| Corn | C4 plant | Terrestrial |
| Cyanobacteria | Prokaryote | Aquatic |



In [7]:
# Same content, different output formats — you get structure for free
for fmt in ['html', 'lml-aggressive', 'lml-compact']:
    out = skillforge.render(ir_json, fmt)
    print(f'{fmt:20s} {len(out):>6d} bytes — preview: {out[:80]!r}')
    print()

html                   1190 bytes — preview: '<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Photos'

lml-aggressive          804 bytes — preview: '#_aif_import_mode: generic\n#_aif_source_format: markdown\n#title: Photosynthesis\n'

lml-compact             893 bytes — preview: '[DOC _aif_import_mode=generic _aif_source_format=markdown title=Photosynthesis]\n'



## 3 · PDF → AIF → PDF / Markdown

PDF support lives in the `aif-pdf` Rust crate (uses `krilla` for export, `pdf_oxide` for import). It's not in the Python bindings to keep the wheel small, so we drive it via the `aif` CLI.

This section:

1. Fetches the Wikipedia PDF of **Quantum computing**
2. Imports it to AIF JSON IR (extracts text, classifies blocks as paragraphs, headings, code)
3. Renders back to PDF
4. Renders to Markdown to show structure preservation

In [8]:
# Locate the aif CLI — either on $PATH or in target/release
import shutil
AIF = shutil.which('aif')
if AIF is None:
    candidate = Path.cwd()
    while candidate != candidate.parent:
        p = candidate / 'target' / 'release' / 'aif'
        if p.exists():
            AIF = str(p); break
        candidate = candidate.parent

assert AIF, 'aif CLI not found — run: cargo build --release -p aif-cli'
print(f'Using CLI: {AIF}')

Using CLI: /Users/liqunchen/Claude_Project/token_efficient/target/release/aif


In [9]:
# Fetch Quantum computing PDF from Wikipedia
pdf_bytes = fetch('https://en.wikipedia.org/api/rest_v1/page/pdf/Quantum_computing')
src_pdf = WORK / 'quantum.pdf'
src_pdf.write_bytes(pdf_bytes)
print(f'Downloaded PDF: {fmt_bytes(len(pdf_bytes))}')

Downloaded PDF: 1.01 MB


In [10]:
# PDF -> AIF JSON IR (via CLI)
json_path = WORK / 'quantum.ir.json'
out = subprocess.run(
    [AIF, 'import', str(src_pdf), '-o', str(json_path)],
    capture_output=True, text=True, check=True
)
print('CLI:', out.stdout.strip(), out.stderr.strip())

ir = json.loads(json_path.read_text())
print(f'\nAIF IR: {len(ir["blocks"])} blocks, metadata: {list(ir["metadata"].keys())}')
print(f'Import confidence: {ir["metadata"].get("_aif_import_confidence", "—")}')
from collections import Counter
kinds = Counter()
for b in ir['blocks']:
    k = b['kind']
    label = k.get('block_type') or k.get('type')
    kinds[label] += 1
print('\nBlock types:')
for t, n in kinds.most_common():
    print(f'  {t:20s} {n}')

CLI:  Imported 34 pages, 157 blocks, avg confidence: 0.76
Wrote /tmp/aif_roundtrip_demo/quantum.ir.json

AIF IR: 157 blocks, metadata: ['_aif_import_confidence', '_aif_source_file', '_aif_source_format', 'title']
Import confidence: 0.76

Block types:
  Paragraph            119
  Section              38


In [11]:
# AIF JSON -> PDF (roundtrip)
rt_pdf = WORK / 'quantum.roundtrip.pdf'
subprocess.run(
    [AIF, 'compile', '--input-format', 'json', str(json_path), '-f', 'pdf', '-o', str(rt_pdf)],
    capture_output=True, check=True
)
print(f'Original PDF:    {fmt_bytes(src_pdf.stat().st_size)}')
print(f'Roundtrip PDF:   {fmt_bytes(rt_pdf.stat().st_size)} ({(1-rt_pdf.stat().st_size/src_pdf.stat().st_size)*100:.0f}% smaller — Wikipedia PDFs carry heavy styling + images that AIF drops)')

Original PDF:    1.01 MB
Roundtrip PDF:   123.9 KB (88% smaller — Wikipedia PDFs carry heavy styling + images that AIF drops)


In [12]:
# AIF JSON -> Markdown (the typical LLM-ingestion path)
out_md = subprocess.run(
    [AIF, 'compile', '--input-format', 'json', str(json_path), '-f', 'markdown'],
    capture_output=True, text=True, check=True
).stdout
print(f'Markdown from PDF: {len(out_md):,} bytes\n')
print('--- first 500 chars ---')
print(out_md[:500])

Markdown from PDF: 115,417 bytes

--- first 500 chars ---
# Quantum computing

## Quantum computing

A quantum computer is a (real or theoretical) computer that exploits quantum phenomena like superposition and entanglement in an essential way. It is widely believed that a quantum computer could perform some calculations exponentially faster than any classical computer. For example, a large-scale quantum computer could break some widely used encryption schemes and aid physicists in performing physical simulations. However, current hardware implementati


In [13]:
# AIF JSON -> LML Aggressive (most token-efficient LLM format)
out_lml = subprocess.run(
    [AIF, 'compile', '--input-format', 'json', str(json_path), '-f', 'lml-aggressive'],
    capture_output=True, text=True, check=True
).stdout

print(f'PDF -> AIF -> LML Aggressive size comparison:')
print(f'  Original PDF:       {fmt_bytes(src_pdf.stat().st_size)}')
print(f'  Markdown (extract): {len(out_md):,} bytes ({(1-len(out_md)/src_pdf.stat().st_size)*100:.0f}% smaller than PDF)')
print(f'  LML Aggressive:     {len(out_lml):,} bytes ({(1-len(out_lml)/src_pdf.stat().st_size)*100:.0f}% smaller)')
print()
print('--- first 400 chars of LML Aggressive ---')
print(out_lml[:400])

PDF -> AIF -> LML Aggressive size comparison:
  Original PDF:       1.01 MB
  Markdown (extract): 115,417 bytes (89% smaller than PDF)
  LML Aggressive:     115,457 bytes (89% smaller)

--- first 400 chars of LML Aggressive ---
#_aif_import_confidence: 0.76
#_aif_source_file: /tmp/aif_roundtrip_demo/quantum.pdf
#_aif_source_format: pdf
#title: Quantum computing
# Quantum computing
A quantum computer is a (real or theoretical) computer that exploits quantum phenomena like superposition and entanglement in an essential way. It is widely believed that a quantum computer could perform some calculations exponentially faster t


## Summary

| Path | Tool | Use case |
|------|------|----------|
| HTML → AIF → * | Python `skillforge.clean_html`, `.render()` | Ingest web pages, strip chrome |
| Markdown → AIF → MD | Python `skillforge.import_markdown`, `.render()` | Convert GitHub-flavored MD through typed IR |
| PDF → AIF → PDF/MD | CLI `aif import` / `aif compile` | Extract structured text from PDFs |

**Key insight:** once content is in AIF IR (JSON), you can render it to any format — HTML, Markdown, LML, PDF, JSON. The single import step does the hard work of extracting structure.

**For LLM ingestion**, `lml-aggressive` typically gives the smallest token footprint while keeping typed block structure. For human editing, `markdown` is usually the right target.

All generated files are in `/tmp/aif_roundtrip_demo/` — inspect them to see the full output.